In [79]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False
import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LogisticRegression 
from sklearn.metrics import mean_absolute_error   # 평균절대값오차 오차범위
from sklearn.preprocessing import PolynomialFeatures,StandardScaler  
from sklearn.model_selection import train_test_split
from scipy.special import expit, softmax    # expit:이진분류,softmax:다중분류 0-1 숫자로 변환
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [80]:
# 로지스틱 회귀 - 다중분류 모델을 이용해서 생선을 분류하시오.
# 다중 : 소프트맥스 사용해서 퍼센트를 구하시오.
# 무게,길이,대각선,높이,넓이를 입력해서 생선을 분류해서 출력하시오.

In [81]:
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.52,4.0200
1,Bream,290.0,26.3,31.2,12.48,4.3056


In [82]:
# 2. data, label분리
import pandas as pd
df = pd.read_csv('data/fish_logistic.csv')
print(df.info())
print(df.head(10))
print(df['Species'].value_counts() if 'Species' in df.columns else df.iloc[:, 0].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    str    
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), str(1)
memory usage: 7.6 KB
None
  Species  Weight  Length  Diagonal   Height   Width
0   Bream   242.0    25.4      30.0  11.5200  4.0200
1   Bream   290.0    26.3      31.2  12.4800  4.3056
2   Bream   340.0    26.5      31.1  12.3778  4.6961
3   Bream   363.0    29.0      33.5  12.7300  4.4555
4   Bream   430.0    29.0      34.0  12.4440  5.1340
5   Bream   450.0    29.7      34.7  13.6024  4.9274
6   Bream   500.0    29.7      34.5  14.1795  5.2785
7   Bream   390.0    30.0      35.0  12.6700  4.6900
8   Bream   450.0    30.0      35.1  14.0049  4.8438
9   Bream   

In [83]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.special import softmax
import zipfile
import os

# 1. 원본 데이터 로드
df = pd.read_csv('data/fish_logistic.csv')

# 2. 판다스에 입력 가능한 각각의 파일들로 분리 저장
# (1) 학습용 / 테스트용 / 예측용(무게,길이,대각선,높이,넓이 입력용) 파일
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['Species'])
train_df.to_csv('fish_train.csv', index=False, encoding='utf-8-sig')
test_df.to_csv('fish_test.csv', index=False, encoding='utf-8-sig')

# 예측 입력용 샘플 파일 (특성 5개만 포함)
predict_input_df = test_df[['Weight', 'Length', 'Diagonal', 'Height', 'Width']].head(5).reset_index(drop=True)
predict_input_df.to_csv('fish_predict_input.csv', index=False, encoding='utf-8-sig')

# (2) 어종(Species)별 각각의 CSV 파일
species_files = []
for species, group in df.groupby('Species'):
    fname = f'fish_{species}.csv'
    group.to_csv(fname, index=False, encoding='utf-8-sig')
    species_files.append(fname)

# 모든 분리 파일을 한 번에 받을 수 있는 ZIP 파일도 생성
all_files = ['fish_train.csv', 'fish_test.csv', 'fish_predict_input.csv'] + species_files
with zipfile.ZipFile('fish_split_files.zip', 'w') as zf:
    for f in all_files:
        zf.write(f)

print("생성된 파일 목록:", all_files)

# 3. 분리된 파일들을 판다스로 다시 읽어 로지스틱 회귀 다중분류 학습 및 예측 실행
train_loaded = pd.read_csv('fish_train.csv')
test_loaded = pd.read_csv('fish_test.csv')
new_fish = pd.read_csv('fish_predict_input.csv')

feature_cols = ['Weight', 'Length', 'Diagonal', 'Height', 'Width']
X_train = train_loaded[feature_cols].to_numpy()
y_train = train_loaded['Species'].to_numpy()
X_test = test_loaded[feature_cols].to_numpy()
y_test = test_loaded['Species'].to_numpy()

# 표준화 스케일링
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)

# 로지스틱 회귀 다중분류 모델 학습
lr = LogisticRegression(C=20, max_iter=1000)
lr.fit(X_train_scaled, y_train)

print(f"\n훈련 세트 정확도: {lr.score(X_train_scaled, y_train):.4f}")
print(f"테스트 세트 정확도: {lr.score(X_test_scaled, y_test):.4f}")
print("클래스 순서:", lr.classes_)

# 4. 새로운 생선 데이터(무게, 길이, 대각선, 높이, 넓이) 입력 후 소프트맥스로 확률(%) 계산 및 분류 출력
X_new_scaled = ss.transform(new_fish[feature_cols].to_numpy())
decision = lr.decision_function(X_new_scaled)
proba = softmax(decision, axis=1)  # 또는 lr.predict_proba(X_new_scaled)
preds = lr.predict(X_new_scaled)

proba_percent_df = pd.DataFrame(np.round(proba * 100, 2), columns=[f"{c}(%)" for c in lr.classes_])
result_df = pd.concat([new_fish, pd.Series(preds, name='예측_생선'), proba_percent_df], axis=1)
print("\n[예측 및 소프트맥스 확률(%) 결과]")
print(result_df.to_string(index=False))

생성된 파일 목록: ['fish_train.csv', 'fish_test.csv', 'fish_predict_input.csv', 'fish_Bream.csv', 'fish_Parkki.csv', 'fish_Perch.csv', 'fish_Pike.csv', 'fish_Roach.csv', 'fish_Smelt.csv', 'fish_Whitefish.csv']

훈련 세트 정확도: 0.9244
테스트 세트 정확도: 0.9750
클래스 순서: ['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']

[예측 및 소프트맥스 확률(%) 결과]
 Weight  Length  Diagonal  Height  Width  예측_생선  Bream(%)  Parkki(%)  Perch(%)  Pike(%)  Roach(%)  Smelt(%)  Whitefish(%)
  120.0    21.0      23.7  6.1146 3.2943  Roach      0.02       2.94     23.70     0.33     68.52      0.98          3.50
  130.0    21.3      22.8  6.3840 3.5340  Perch      0.02       3.19     57.65     0.12     34.90      0.27          3.86
   80.0    19.0      20.2  5.6358 3.0502  Perch      0.01       6.19     55.86     0.08     33.52      1.72          2.62
  120.0    19.0      21.3  8.3922 2.9181 Parkki      0.30      92.98      0.09     0.00      5.12      0.03          1.48
   90.0    17.7      19.8  7.4052 2.6730 Parkki      0.0